## Modeling

In [2]:
DATA_FILE = "Data/all_dialogs.txt"

all_dialogs = open(DATA_FILE)

text = all_dialogs.read()

dialogs = text.split("-" * 100)

print(len(dialogs))

3236


In [3]:
#print(dialogs[0])

## Implementation

In [4]:
dialog_state = {
    "state": 1,
    "area": None,
    "food": None,
    "pricerange": None,
    "restaurant": None,
    "alternatives": []
}

### Slot extraction

In [5]:
import pandas as pd

RESTAURANT_FILE = "Data/restaurant_info.csv"

restaurants = pd.read_csv(RESTAURANT_FILE)

#print(restaurants.columns)
#restaurants.head()

In [6]:
areas = restaurants["area"].dropna().unique().tolist()
foods = restaurants["food"].dropna().unique().tolist()
price_ranges = restaurants["pricerange"].dropna().unique().tolist()

#print(areas)
#print(price_ranges)
#print(foods)

In [14]:
### 1. Keyword matching

def keyword_slot_extractor(utterance):
    utterance = utterance.lower()
    
    slots = {
        "area": None,
        "food": None,
        "pricerange": None
    }
    
    for area in areas:
        if area in utterance:
            slots["area"] = area

    for food in foods:
        if food in utterance:
            slots["food"] = food

    for price in price_ranges:
        if price in utterance:
            slots["pricerange"] = price
            
    
    return slots

In [15]:
### 2. Levenshtein edit distance

#%pip install levenshtein
from Levenshtein import distance

def levenshtein_slot_extractor(utterance):
    utterance = utterance.lower()
    words = utterance.split()

    slots = {
        "area": None,
        "food": None,
        "pricerange": None
    }
    
    # Area
    best_match = None
    best_distance = float("inf")

    for area in areas:
        for word in words:
            current_distance = distance(word, area)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = area

    slots["area"] = best_match        
    
    
    # Food
    best_match = None
    best_distance = float("inf")

    for food in foods:
        for word in words:
            current_distance = distance(word, food)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = food

    slots["food"] = best_match
    

    # Pricerange
    best_match = None
    best_distance = float("inf")

    for price in price_ranges:
        for word in words:
            current_distance = distance(word, price)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = price

    slots["pricerange"] = best_match

    return slots

In [16]:
### 3. Semantic similarity via embeddings

## TODO: Alessandro

### Response generation

Use string templates for all system utterances, either simple strings or patterns with variables to fill in. For example:

"askpricerange": "Which price range do you prefer?"
"confirmfoodtype": "I did not recognize {givenfoodtype}. Did you mean {correctedfoodtype}?"

In [10]:
## TODO: Niousha 

def response_generation(system_utterance, restaurant=None, food=None, area=None, pricerange=None):
    


### Restaurant lookup

Once all preferences are collected, the system provides a recommendation. Write a Python function that retrieves matching restaurants from restaurant_info.csv according to the extracted preferences. If no restaurant matches, the system should inform the user. If multiple match, one is chosen at random and the others are stored so the user can ask for an alternative using the same preferences.

In [17]:
import random
import pandas as pd

restaurants = pd.read_csv("Data/restaurant_info.csv")

def restaurant_lookup(area, food, pricerange):

    matches = restaurants[(restaurants["area"] == area) & (restaurants["food"] == food) & (restaurants["pricerange"] == pricerange)]
    matches = matches["restaurantname"].tolist()

    if len(matches) == 0:
        return None, []
    
    
    recommendation = random.choice(matches)
    
    alternatives = [restaurant for restaurant in matches if restaurant != recommendation]
        
    return recommendation, alternatives


#recommendation, alternatives = restaurant_lookup("north", "indian", "moderate")
#print(recommendation)
#print(alternatives)

### Reasoning component

In [12]:
RESTAURANT_FILE2 = "Data/restaurant_info_extended.csv"

restaurants_extended = pd.read_csv(RESTAURANT_FILE2)

consequents=["touristic", "assigned seats", "children", "romantic"]

def antecedent_rules(consequent):
    if consequent == "touristic":
        true={"pricerange": "cheap",
             "food quality": "good"}
        false= {"food": "romanian"}
    elif consequent == "assigned seats":
        true={"crowdedness": "busy"}
        false= {}
    elif consequent == "children":
        true={}
        false={"length of stay": "long"}
    elif consequent == "romantic":
        true={"crowdedness": "busy"}
        false= {"length of stay": "long"}
    return true, false

def reasoning_component(restaurants_options, consequent):
    true, false = antecedent_rules(consequent)
    matches=[]
    half_match=[]

    for restaurant_option in restaurants_options:
            restaurant = restaurants_extended[restaurants_extended["restaurantname"] == restaurant_option]
            restaurant = restaurant.iloc[0]
        
            positive_match = all(restaurant[attribute] == value
                                 for attribute, value in true.items())
        
            negative_match = all(restaurant[attribute] != value
                                 for attribute, value in false.items())

            if positive_match and negative_match:
                matches.append(restaurant)

            recommendation = random.choice(matches)
        
    return recommendation["restaurantname"]

            # contradiction:
            # if positive_match and not(negative_match):
            #     half_match.append(restaurant)
            
    #return half_match

# recommendation=['frankie and bennys']
# candidates=['the nirala', 'tandoori palace', 'graffiti']
# con="romantic"

# matchings = reasoning_component(candidates, con)

# print(matchings)


## Implementation / Configurability

In [13]:
# state transition function
def state_transition(dialog_state, utterance, dialog_act):
    
    current_state = dialog_state["state"]
    
    if dialog_act == "inform":
        
        slots = keyword_slot_extractor(utterance)
        
        if slots["food"] is not None:
            dialog_state["food"] = slots["food"]

        if slots["area"] is not None:
            dialog_state["area"] = slots["area"]

        if slots["pricerange"] is not None:
            dialog_state["pricerange"] = slots["pricerange"]
            
        
        if dialog_state["food"] is None:
            next_dialog_state = 2                                    # if 2 is ask food type
            system_utterance = response_generation("askfoodtype")

        elif dialog_state["area"] is None:
            next_dialog_state = 3                                    # if 3 is ask area
            system_utterance = response_generation("askarea")

        elif dialog_state["pricerange"] is None:
            next_dialog_state = 4                                    # if 4 if ask price range
            system_utterance = response_generation("askpricerange")

        else:
            
            recommendation, alternatives = restaurant_lookup(dialog_state["area"], dialog_state["food"], dialog_state["pricerange"])
    
            if recommendation is not None:
                dialog_state["restaurant"] = recommendation
                dialog_state["alternatives"] = alternatives
                
                next_dialog_state = 5                               # if 5 is recommend restaurant
                system_utterance = response_generation("recommendrestaurant", recommendation)
                
            else:
                next_dialog_state = 6                              # if 6 is no match
                system_utterance = response_generation("nomatch")
                
                
                
        dialog_state["state"] = next_dialog_state
                
    
    return(next_dialog_state, system_utterance)